# Engenharia de Dados — Transformação, Qualidade, Carga e Data Warehouse

**Curso:** Pós-Graduação em Ciência de Dados  
**Disciplina:** Engenharia de Dados (ETL) e Big Data  
**Blocos:** Transformação e Qualidade dos Dados; Carga e Data Warehouse  
**Ambiente:** Google Colab

O notebook utiliza dados sintéticos de uma instituição de ensino. Ele demonstra um pipeline completo, desde a avaliação da qualidade até a construção de um Data Warehouse em SQLite.

> Nenhum dado pessoal real é utilizado.

## Objetivos

Ao final da aula, o estudante deverá conseguir:

1. Limpar, padronizar, converter e validar dados;
2. Tratar nulos e duplicidades com justificativa;
3. Integrar fontes sem alterar indevidamente o grão;
4. Aplicar regras de negócio e enriquecimentos;
5. Diferenciar carga completa e incremental;
6. Declarar o grão de uma tabela fato;
7. Construir dimensões e fatos com chaves substitutas;
8. Comparar esquema estrela e floco de neve;
9. Executar e auditar uma carga em Data Warehouse.

### Fluxo

**Fontes → Qualidade → Transformação → Integração → Regras de negócio → Dimensões → Fato → Carga → Auditoria → Consumo**

## 1. Preparação do ambiente

In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import json
import re
import sqlite3
import unicodedata

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

pd.set_option("display.max_columns", 50)
pd.set_option("display.max_colwidth", 80)

SEMENTE = 2026
gerador = np.random.default_rng(SEMENTE)

PASTA_BASE = Path("dados_tarde")
PASTA_BRONZE = PASTA_BASE / "bronze"
PASTA_SILVER = PASTA_BASE / "silver"
PASTA_GOLD = PASTA_BASE / "gold"
PASTA_QUARENTENA = PASTA_BASE / "quarentena"

for pasta in [PASTA_BRONZE, PASTA_SILVER, PASTA_GOLD, PASTA_QUARENTENA]:
    pasta.mkdir(parents=True, exist_ok=True)

CAMINHO_DW = PASTA_GOLD / "dw_academico.db"

print("Ambiente preparado.")
print(f"Data Warehouse: {CAMINHO_DW.resolve()}")

## 2. Fontes operacionais

Usaremos duas fontes com grãos diferentes:

- `matriculas.csv`: uma linha por matrícula;
- `pagamentos.json`: uma linha por pagamento.

Antes da integração, os pagamentos precisam ser agregados por matrícula. Uma junção direta poderia multiplicar linhas e alterar o grão da futura tabela fato.

In [ ]:
# Dados de matrículas com inconsistências intencionais.
nomes = [
    "Ana Souza", "Bruno Lima", "Carla Mendes", "Diego Santos", "Elisa Rocha", "Felipe Alves",
    "Gabriela Silva", "Henrique Costa", "Isabela Nunes", "João Melo", "Karina Freitas", "Lucas Barros",
    "Mariana Oliveira", "Nicolas Araújo", "Paula Ribeiro", "Rafael Gomes", "Sabrina Martins", "Tiago Ferreira",
    "Valéria Castro", "William Lopes", "Aline Cavalcanti", "Caio Monteiro", "Débora Almeida", "Eduardo Correia"
]

cidades = ["recife", "RECIFE", "Olinda", "olinda ", "Paulista", "PAULISTA", "JABOATÃO"]
cursos = ["Ciência de Dados", "ciencia de dados", "CD", "Engenharia de Software", "eng. software", "IA", "Inteligência Artificial"]
status = ["Ativa", "ATIVO", "Pendente", "pendente", "Cancelada", "CANCELADO"]

linhas = []
for indice, nome in enumerate(nomes, start=1):
    data = pd.Timestamp("2026-07-01") + pd.Timedelta(days=int(gerador.integers(0, 75)))
    mensalidade = float(gerador.choice([900, 1050, 1200, 1350]))
    desconto = float(gerador.choice([0, 5, 10, 15, 20]))

    linhas.append(
        {
            "matricula_id": 5000 + indice,
            "aluno_id": 1000 + indice,
            "nome": nome if indice % 5 else f"  {nome.upper()} ",
            "idade": int(gerador.integers(21, 58)),
            "email": f"{nome.lower().replace(' ', '.')}@exemplo.com",
            "cidade": gerador.choice(cidades),
            "curso": gerador.choice(cursos),
            "status_matricula": gerador.choice(status),
            "data_matricula": data.strftime("%d/%m/%Y") if indice % 2 else data.strftime("%Y-%m-%d"),
            "mensalidade": mensalidade if indice % 4 else f"R$ {mensalidade:,.2f}".replace(",", "X").replace(".", ",").replace("X", "."),
            "desconto_pct": desconto,
            "atualizado_em": (data + pd.Timedelta(days=2)).strftime("%Y-%m-%d %H:%M:%S"),
        }
    )

matriculas_origem = pd.DataFrame(linhas)

# Problemas intencionais para discussão.
matriculas_origem.loc[2, "idade"] = None
matriculas_origem.loc[6, "idade"] = 130
matriculas_origem.loc[4, "email"] = None
matriculas_origem.loc[9, "email"] = "email_invalido"
matriculas_origem.loc[11, "mensalidade"] = None
matriculas_origem.loc[14, "desconto_pct"] = 80
matriculas_origem.loc[18, "data_matricula"] = "31/02/2026"

# Duplicidade exata e atualização posterior da mesma matrícula.
matriculas_origem = pd.concat([matriculas_origem, matriculas_origem.iloc[[5]]], ignore_index=True)
atualizacao = matriculas_origem.iloc[[1]].copy()
atualizacao["status_matricula"] = "Ativa"
atualizacao["atualizado_em"] = "2026-09-18 15:00:00"
matriculas_origem = pd.concat([matriculas_origem, atualizacao], ignore_index=True)

caminho_matriculas = PASTA_BRONZE / "matriculas.csv"
matriculas_origem.to_csv(caminho_matriculas, index=False, encoding="utf-8")

# Pagamentos: várias linhas podem pertencer à mesma matrícula.
pagamentos = []
pagamento_id = 1
for matricula_id in range(5001, 5025):
    quantidade = int(gerador.integers(0, 4))
    for parcela in range(quantidade):
        pagamentos.append(
            {
                "pagamento_id": pagamento_id,
                "matricula_id": matricula_id,
                "data_pagamento": (pd.Timestamp("2026-08-01") + pd.Timedelta(days=parcela * 30)).strftime("%Y-%m-%d"),
                "valor_pago": float(gerador.choice([800, 900, 1000, 1100, 1200])),
                "situacao": gerador.choice(["Pago", "pago", "Atrasado"]),
                "origem": {"canal": gerador.choice(["Portal", "PIX", "Boleto"]), "lote": "2026-09"},
            }
        )
        pagamento_id += 1

# Pagamento órfão: não existe matrícula correspondente.
pagamentos.append(
    {
        "pagamento_id": 999,
        "matricula_id": 9999,
        "data_pagamento": "2026-09-15",
        "valor_pago": 1000,
        "situacao": "Pago",
        "origem": {"canal": "PIX", "lote": "2026-09"},
    }
)

caminho_pagamentos = PASTA_BRONZE / "pagamentos.json"
with open(caminho_pagamentos, "w", encoding="utf-8") as arquivo:
    json.dump(pagamentos, arquivo, ensure_ascii=False, indent=2)

print(f"Matrículas recebidas: {len(matriculas_origem)}")
print(f"Pagamentos recebidos: {len(pagamentos)}")

## 3. Extração e diagnóstico de qualidade

In [ ]:
matriculas_brutas = pd.read_csv(caminho_matriculas, dtype="object")

with open(caminho_pagamentos, "r", encoding="utf-8") as arquivo:
    pagamentos_brutos = pd.json_normalize(json.load(arquivo), sep="_")


def perfil_qualidade(nome, df, chave=None):
    """Produz um perfil compacto para orientar as decisões de tratamento."""
    resultado = {
        "fonte": nome,
        "linhas": len(df),
        "colunas": len(df.columns),
        "celulas_nulas": int(df.isna().sum().sum()),
        "duplicidades_exatas": int(df.duplicated().sum()),
    }
    if chave:
        resultado["duplicidades_chave"] = int(df.duplicated(subset=chave).sum())
    return resultado


resumo_qualidade = pd.DataFrame(
    [
        perfil_qualidade("matriculas", matriculas_brutas, ["matricula_id"]),
        perfil_qualidade("pagamentos", pagamentos_brutos, ["pagamento_id"]),
    ]
)

display(resumo_qualidade)
display(matriculas_brutas.head(8))
display(pagamentos_brutos.head(8))

print("Ausências nas matrículas:")
display(matriculas_brutas.isna().sum().rename("quantidade").to_frame().query("quantidade > 0"))

## 4. Funções de limpeza e padronização

Centralizar regras em funções evita tratamentos diferentes para o mesmo conceito. As funções também serão reutilizadas na carga incremental e na atividade.

In [ ]:
def sem_acentos(valor):
    if pd.isna(valor):
        return None
    normalizado = unicodedata.normalize("NFKD", str(valor))
    return "".join(c for c in normalizado if not unicodedata.combining(c))


def chave_textual(valor):
    if pd.isna(valor):
        return None
    return " ".join(sem_acentos(valor).strip().lower().split())


def nome_padronizado(valor):
    if pd.isna(valor):
        return pd.NA
    return " ".join(str(valor).strip().split()).title()


MAPA_CIDADE = {
    "recife": "Recife",
    "olinda": "Olinda",
    "paulista": "Paulista",
    "jaboatao": "Jaboatão dos Guararapes",
}

MAPA_CURSO = {
    "ciencia de dados": "Ciência de Dados",
    "cd": "Ciência de Dados",
    "engenharia de software": "Engenharia de Software",
    "eng. software": "Engenharia de Software",
    "inteligencia artificial": "Inteligência Artificial",
    "ia": "Inteligência Artificial",
}

MAPA_STATUS = {
    "ativa": "Ativa", "ativo": "Ativa",
    "pendente": "Pendente",
    "cancelada": "Cancelada", "cancelado": "Cancelada",
}


def padronizar_categoria(valor, mapa, padrao="Não informado"):
    return mapa.get(chave_textual(valor), padrao)


def moeda_para_float(valor):
    if pd.isna(valor) or str(valor).strip() == "":
        return np.nan
    texto = str(valor).replace("R$", "").replace(" ", "")
    if "." in texto and "," in texto:
        texto = texto.replace(".", "").replace(",", ".")
    elif "," in texto:
        texto = texto.replace(",", ".")
    return pd.to_numeric(texto, errors="coerce")


def email_valido(valor):
    if pd.isna(valor):
        return False
    return bool(re.match(r"^[^\s@]+@[^\s@]+\.[^\s@]+$", str(valor).strip()))


def limpar_matriculas(df, referencia_historica=None):
    """Aplica limpeza, conversões, regras e deduplicação por versão mais recente."""
    saida = df.copy()
    saida["matricula_id"] = pd.to_numeric(saida["matricula_id"], errors="coerce").astype("Int64")
    saida["aluno_id"] = pd.to_numeric(saida["aluno_id"], errors="coerce").astype("Int64")
    saida["atualizado_em"] = pd.to_datetime(saida["atualizado_em"], errors="coerce")
    saida["data_matricula"] = pd.to_datetime(saida["data_matricula"], format="mixed", dayfirst=True, errors="coerce")
    saida["nome"] = saida["nome"].apply(nome_padronizado)
    saida["cidade"] = saida["cidade"].apply(lambda x: padronizar_categoria(x, MAPA_CIDADE))
    saida["curso"] = saida["curso"].apply(lambda x: padronizar_categoria(x, MAPA_CURSO))
    saida["status_matricula"] = saida["status_matricula"].apply(lambda x: padronizar_categoria(x, MAPA_STATUS))

    saida["idade"] = pd.to_numeric(saida["idade"], errors="coerce")
    saida.loc[~saida["idade"].between(18, 80), "idade"] = np.nan
    saida["idade_foi_imputada"] = saida["idade"].isna()

    if referencia_historica is not None:
        mediana_idade = float(referencia_historica["idade"].median())
    else:
        mediana_idade = float(saida["idade"].median())
    saida["idade"] = saida["idade"].fillna(mediana_idade).round().astype("Int64")

    saida["mensalidade"] = saida["mensalidade"].apply(moeda_para_float)
    saida["mensalidade_foi_imputada"] = saida["mensalidade"].isna()
    mediana_mensalidade = (
        float(referencia_historica["mensalidade"].median())
        if referencia_historica is not None
        else float(saida["mensalidade"].median())
    )
    saida["mensalidade"] = saida["mensalidade"].fillna(mediana_mensalidade).round(2)

    saida["desconto_pct"] = pd.to_numeric(saida["desconto_pct"], errors="coerce")
    saida["desconto_ajustado"] = ~saida["desconto_pct"].between(0, 50)
    saida["desconto_pct"] = saida["desconto_pct"].clip(lower=0, upper=50).fillna(0)

    saida["email"] = saida["email"].apply(lambda x: str(x).strip().lower() if pd.notna(x) else pd.NA)
    saida["email_valido"] = saida["email"].apply(email_valido)

    saida = (
        saida.drop_duplicates()
        .sort_values(["matricula_id", "atualizado_em"])
        .drop_duplicates("matricula_id", keep="last")
        .reset_index(drop=True)
    )
    return saida


print("Funções carregadas.")

## 5. Transformação, validação e quarentena

In [ ]:
matriculas_limpas = limpar_matriculas(matriculas_brutas)

# Registros sem chaves ou data válida não seguem para o Data Warehouse.
mascara_quarentena = (
    matriculas_limpas["matricula_id"].isna()
    | matriculas_limpas["aluno_id"].isna()
    | matriculas_limpas["data_matricula"].isna()
)

matriculas_quarentena = matriculas_limpas[mascara_quarentena].copy()
matriculas_validas = matriculas_limpas[~mascara_quarentena].copy()

matriculas_quarentena.to_csv(PASTA_QUARENTENA / "matriculas_rejeitadas.csv", index=False)

# Transformação da fonte de pagamentos.
pagamentos_limpios = pagamentos_brutos.copy()
pagamentos_limpios["pagamento_id"] = pd.to_numeric(pagamentos_limpios["pagamento_id"], errors="coerce").astype("Int64")
pagamentos_limpios["matricula_id"] = pd.to_numeric(pagamentos_limpios["matricula_id"], errors="coerce").astype("Int64")
pagamentos_limpios["data_pagamento"] = pd.to_datetime(pagamentos_limpios["data_pagamento"], errors="coerce")
pagamentos_limpios["valor_pago"] = pd.to_numeric(pagamentos_limpios["valor_pago"], errors="coerce")
pagamentos_limpios["situacao"] = pagamentos_limpios["situacao"].apply(nome_padronizado)
pagamentos_limpios = pagamentos_limpios.drop_duplicates("pagamento_id", keep="last")

# Anti-join para localizar pagamentos órfãos.
ids_matriculas = set(matriculas_validas["matricula_id"].astype(int))
pagamentos_orfaos = pagamentos_limpios[~pagamentos_limpios["matricula_id"].isin(ids_matriculas)].copy()

print(f"Matrículas válidas: {len(matriculas_validas)}")
print(f"Matrículas em quarentena: {len(matriculas_quarentena)}")
print(f"Pagamentos órfãos: {len(pagamentos_orfaos)}")
display(matriculas_quarentena)
display(pagamentos_orfaos)

## 6. Integração, regras de negócio e enriquecimento

O grão será preservado como **uma linha por matrícula**. Por isso, pagamentos são agregados antes da junção.

In [ ]:
pagamentos_por_matricula = (
    pagamentos_limpios[pagamentos_limpios["matricula_id"].isin(ids_matriculas)]
    .groupby("matricula_id", as_index=False)
    .agg(
        quantidade_pagamentos=("pagamento_id", "count"),
        total_pago=("valor_pago", "sum"),
        ultimo_pagamento=("data_pagamento", "max"),
        canais_pagamento=("origem_canal", lambda x: ", ".join(sorted(set(x)))),
    )
)

dados_enriquecidos = matriculas_validas.merge(
    pagamentos_por_matricula,
    on="matricula_id",
    how="left",
    validate="one_to_one",
)

dados_enriquecidos["quantidade_pagamentos"] = dados_enriquecidos["quantidade_pagamentos"].fillna(0).astype(int)
dados_enriquecidos["total_pago"] = dados_enriquecidos["total_pago"].fillna(0.0)
dados_enriquecidos["canais_pagamento"] = dados_enriquecidos["canais_pagamento"].fillna("Nenhum")

# Regras de negócio e atributos derivados.
dados_enriquecidos["valor_liquido"] = (
    dados_enriquecidos["mensalidade"] * (1 - dados_enriquecidos["desconto_pct"] / 100)
).round(2)

dados_enriquecidos["faixa_etaria"] = pd.cut(
    dados_enriquecidos["idade"].astype(float),
    bins=[17, 25, 35, 50, 80],
    labels=["18–25", "26–35", "36–50", "51–80"],
)

MAPA_REGIAO = {
    "Recife": "Núcleo metropolitano",
    "Olinda": "Região Metropolitana",
    "Paulista": "Região Metropolitana",
    "Jaboatão dos Guararapes": "Região Metropolitana",
}
dados_enriquecidos["regiao"] = dados_enriquecidos["cidade"].map(MAPA_REGIAO).fillna("Não informada")

dados_enriquecidos["risco_financeiro"] = np.select(
    [
        dados_enriquecidos["status_matricula"].eq("Cancelada"),
        dados_enriquecidos["quantidade_pagamentos"].eq(0),
        dados_enriquecidos["total_pago"].lt(dados_enriquecidos["valor_liquido"]),
    ],
    ["Encerrado", "Alto", "Médio"],
    default="Baixo",
)

# Regras executáveis.
assert dados_enriquecidos["matricula_id"].is_unique
assert dados_enriquecidos["idade"].between(18, 80).all()
assert dados_enriquecidos["mensalidade"].ge(0).all()
assert dados_enriquecidos["desconto_pct"].between(0, 50).all()
assert dados_enriquecidos["valor_liquido"].ge(0).all()

caminho_silver = PASTA_SILVER / "matriculas_enriquecidas.csv"
dados_enriquecidos.to_csv(caminho_silver, index=False)

display(
    dados_enriquecidos[[
        "matricula_id", "nome", "curso", "status_matricula", "mensalidade",
        "desconto_pct", "valor_liquido", "total_pago", "risco_financeiro"
    ]].head(10)
)

## 7. Modelagem dimensional

### Grão declarado

**Uma linha na `fato_matricula` representa uma matrícula de um aluno em um curso, na data da matrícula.**

### Esquema estrela adotado

- `dim_aluno`: atributos do aluno, incluindo cidade e região;
- `dim_curso`: atributos do curso;
- `dim_data`: calendário;
- `fato_matricula`: medidas da matrícula e dos pagamentos agregados.

As dimensões usam chaves substitutas (`*_sk`). As chaves naturais permanecem para integração e auditoria.

In [ ]:
def criar_estrutura_dw(conexao):
    """Cria as tabelas do Data Warehouse e as restrições necessárias ao upsert."""
    conexao.executescript(
        """
        PRAGMA foreign_keys = ON;

        CREATE TABLE IF NOT EXISTS dim_aluno (
            aluno_sk INTEGER PRIMARY KEY AUTOINCREMENT,
            aluno_id INTEGER NOT NULL UNIQUE,
            nome TEXT NOT NULL,
            idade INTEGER,
            faixa_etaria TEXT,
            email TEXT,
            email_valido INTEGER NOT NULL,
            cidade TEXT,
            regiao TEXT,
            atualizado_em TEXT NOT NULL
        );

        CREATE TABLE IF NOT EXISTS dim_curso (
            curso_sk INTEGER PRIMARY KEY AUTOINCREMENT,
            curso TEXT NOT NULL UNIQUE
        );

        CREATE TABLE IF NOT EXISTS dim_data (
            data_sk INTEGER PRIMARY KEY,
            data_completa TEXT NOT NULL UNIQUE,
            ano INTEGER NOT NULL,
            mes INTEGER NOT NULL,
            nome_mes TEXT NOT NULL,
            trimestre INTEGER NOT NULL
        );

        CREATE TABLE IF NOT EXISTS fato_matricula (
            matricula_id INTEGER PRIMARY KEY,
            aluno_sk INTEGER NOT NULL,
            curso_sk INTEGER NOT NULL,
            data_sk INTEGER NOT NULL,
            status_matricula TEXT NOT NULL,
            mensalidade REAL NOT NULL,
            desconto_pct REAL NOT NULL,
            valor_liquido REAL NOT NULL,
            quantidade_pagamentos INTEGER NOT NULL,
            total_pago REAL NOT NULL,
            risco_financeiro TEXT NOT NULL,
            atualizado_em TEXT NOT NULL,
            FOREIGN KEY (aluno_sk) REFERENCES dim_aluno(aluno_sk),
            FOREIGN KEY (curso_sk) REFERENCES dim_curso(curso_sk),
            FOREIGN KEY (data_sk) REFERENCES dim_data(data_sk)
        );

        CREATE TABLE IF NOT EXISTS controle_carga (
            carga_id INTEGER PRIMARY KEY AUTOINCREMENT,
            tipo_carga TEXT NOT NULL,
            iniciado_em TEXT NOT NULL,
            finalizado_em TEXT NOT NULL,
            registros_recebidos INTEGER NOT NULL,
            registros_carregados INTEGER NOT NULL,
            registros_rejeitados INTEGER NOT NULL,
            marca_dagua TEXT
        );
        """
    )


def preparar_linhas_dw(df):
    """Transforma o DataFrame Silver em registros prontos para dimensões e fato."""
    linhas = df.copy()
    linhas["data_sk"] = linhas["data_matricula"].dt.strftime("%Y%m%d").astype(int)
    linhas["data_completa"] = linhas["data_matricula"].dt.strftime("%Y-%m-%d")
    linhas["ano"] = linhas["data_matricula"].dt.year
    linhas["mes"] = linhas["data_matricula"].dt.month
    linhas["nome_mes"] = linhas["data_matricula"].dt.month_name()
    linhas["trimestre"] = linhas["data_matricula"].dt.quarter
    return linhas


def carregar_dw(df, tipo_carga="incremental", registros_rejeitados=0):
    """Executa upserts idempotentes nas dimensões e na tabela fato."""
    inicio = datetime.now(timezone.utc).isoformat()
    linhas = preparar_linhas_dw(df)

    with sqlite3.connect(CAMINHO_DW) as conexao:
        criar_estrutura_dw(conexao)

        # Dimensão curso: insere apenas categorias ainda inexistentes.
        conexao.executemany(
            "INSERT INTO dim_curso (curso) VALUES (?) ON CONFLICT(curso) DO NOTHING",
            [(curso,) for curso in sorted(linhas["curso"].dropna().unique())],
        )

        # Dimensão data: uma linha por data de matrícula.
        dados_data = linhas[["data_sk", "data_completa", "ano", "mes", "nome_mes", "trimestre"]].drop_duplicates()
        conexao.executemany(
            """
            INSERT INTO dim_data (data_sk, data_completa, ano, mes, nome_mes, trimestre)
            VALUES (?, ?, ?, ?, ?, ?)
            ON CONFLICT(data_sk) DO UPDATE SET
                data_completa=excluded.data_completa,
                ano=excluded.ano,
                mes=excluded.mes,
                nome_mes=excluded.nome_mes,
                trimestre=excluded.trimestre
            """,
            list(dados_data.itertuples(index=False, name=None)),
        )

        # Dimensão aluno: atualiza atributos quando a chave natural já existe.
        alunos = linhas[[
            "aluno_id", "nome", "idade", "faixa_etaria", "email", "email_valido",
            "cidade", "regiao", "atualizado_em"
        ]].drop_duplicates("aluno_id", keep="last").copy()
        alunos["faixa_etaria"] = alunos["faixa_etaria"].astype(str)
        alunos["email_valido"] = alunos["email_valido"].astype(int)
        alunos["atualizado_em"] = alunos["atualizado_em"].astype(str)
        # SQLite não aceita diretamente o marcador pd.NA; valores ausentes viram NULL.
        alunos = alunos.astype(object).where(pd.notna(alunos), None)

        conexao.executemany(
            """
            INSERT INTO dim_aluno (
                aluno_id, nome, idade, faixa_etaria, email, email_valido, cidade, regiao, atualizado_em
            ) VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?)
            ON CONFLICT(aluno_id) DO UPDATE SET
                nome=excluded.nome,
                idade=excluded.idade,
                faixa_etaria=excluded.faixa_etaria,
                email=excluded.email,
                email_valido=excluded.email_valido,
                cidade=excluded.cidade,
                regiao=excluded.regiao,
                atualizado_em=excluded.atualizado_em
            """,
            list(alunos.itertuples(index=False, name=None)),
        )

        # Recupera as chaves substitutas geradas no destino.
        mapa_aluno = pd.read_sql_query("SELECT aluno_id, aluno_sk FROM dim_aluno", conexao)
        mapa_curso = pd.read_sql_query("SELECT curso, curso_sk FROM dim_curso", conexao)
        fato = (
            linhas.merge(mapa_aluno, on="aluno_id", how="left", validate="many_to_one")
            .merge(mapa_curso, on="curso", how="left", validate="many_to_one")
        )

        dados_fato = fato[[
            "matricula_id", "aluno_sk", "curso_sk", "data_sk", "status_matricula",
            "mensalidade", "desconto_pct", "valor_liquido", "quantidade_pagamentos",
            "total_pago", "risco_financeiro", "atualizado_em"
        ]].copy()
        dados_fato["atualizado_em"] = dados_fato["atualizado_em"].astype(str)

        # Converte escalares NumPy/Pandas para tipos nativos aceitos pelo sqlite3.
        registros_fato = [
            (
                int(linha.matricula_id), int(linha.aluno_sk), int(linha.curso_sk), int(linha.data_sk),
                str(linha.status_matricula), float(linha.mensalidade), float(linha.desconto_pct),
                float(linha.valor_liquido), int(linha.quantidade_pagamentos), float(linha.total_pago),
                str(linha.risco_financeiro), str(linha.atualizado_em),
            )
            for linha in dados_fato.itertuples(index=False)
        ]

        conexao.executemany(
            """
            INSERT INTO fato_matricula (
                matricula_id, aluno_sk, curso_sk, data_sk, status_matricula,
                mensalidade, desconto_pct, valor_liquido, quantidade_pagamentos,
                total_pago, risco_financeiro, atualizado_em
            ) VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?)
            ON CONFLICT(matricula_id) DO UPDATE SET
                aluno_sk=excluded.aluno_sk,
                curso_sk=excluded.curso_sk,
                data_sk=excluded.data_sk,
                status_matricula=excluded.status_matricula,
                mensalidade=excluded.mensalidade,
                desconto_pct=excluded.desconto_pct,
                valor_liquido=excluded.valor_liquido,
                quantidade_pagamentos=excluded.quantidade_pagamentos,
                total_pago=excluded.total_pago,
                risco_financeiro=excluded.risco_financeiro,
                atualizado_em=excluded.atualizado_em
            """,
            registros_fato,
        )

        marca_dagua = linhas["atualizado_em"].max()
        fim = datetime.now(timezone.utc).isoformat()
        conexao.execute(
            """
            INSERT INTO controle_carga (
                tipo_carga, iniciado_em, finalizado_em, registros_recebidos,
                registros_carregados, registros_rejeitados, marca_dagua
            ) VALUES (?, ?, ?, ?, ?, ?, ?)
            """,
            (tipo_carga, inicio, fim, len(df) + registros_rejeitados, len(df), registros_rejeitados, str(marca_dagua)),
        )
        conexao.commit()

    return len(df)


print("Estrutura e funções de carga definidas.")

## 8. Carga completa inicial

In [ ]:
# Em uma carga completa controlada, reconstruímos o destino do exemplo.
if CAMINHO_DW.exists():
    CAMINHO_DW.unlink()

carregados = carregar_dw(
    dados_enriquecidos,
    tipo_carga="completa",
    registros_rejeitados=len(matriculas_quarentena),
)

with sqlite3.connect(CAMINHO_DW) as conexao:
    contagens = pd.read_sql_query(
        """
        SELECT 'dim_aluno' AS tabela, COUNT(*) AS linhas FROM dim_aluno
        UNION ALL SELECT 'dim_curso', COUNT(*) FROM dim_curso
        UNION ALL SELECT 'dim_data', COUNT(*) FROM dim_data
        UNION ALL SELECT 'fato_matricula', COUNT(*) FROM fato_matricula
        UNION ALL SELECT 'controle_carga', COUNT(*) FROM controle_carga
        """,
        conexao,
    )

print(f"Registros carregados na fato: {carregados}")
display(contagens)

## 9. Carga incremental

A carga incremental recebe somente registros novos ou atualizados desde a última marca d’água. O `upsert` torna a operação idempotente: executar o mesmo lote novamente não cria duplicidades na chave de negócio.

In [ ]:
# Um registro atualiza matrícula existente; dois são novas matrículas.
lote_incremental_bruto = pd.DataFrame(
    [
        {
            "matricula_id": 5002, "aluno_id": 1002, "nome": "Bruno Lima", "idade": 32,
            "email": "bruno.lima@exemplo.com", "cidade": "RECIFE", "curso": "CD",
            "status_matricula": "Ativa", "data_matricula": "2026-07-18",
            "mensalidade": 1200, "desconto_pct": 10, "atualizado_em": "2026-09-20 09:00:00",
        },
        {
            "matricula_id": 5025, "aluno_id": 1025, "nome": "Fernanda Sales", "idade": 29,
            "email": "fernanda.sales@exemplo.com", "cidade": "Olinda", "curso": "IA",
            "status_matricula": "Ativa", "data_matricula": "20/09/2026",
            "mensalidade": "R$ 1.350,00", "desconto_pct": 15, "atualizado_em": "2026-09-20 10:00:00",
        },
        {
            "matricula_id": 5026, "aluno_id": 1026, "nome": "Gustavo Pires", "idade": 41,
            "email": "gustavo.pires@exemplo.com", "cidade": "Paulista", "curso": "Engenharia de Software",
            "status_matricula": "Pendente", "data_matricula": "2026-09-20",
            "mensalidade": 1050, "desconto_pct": 0, "atualizado_em": "2026-09-20 11:00:00",
        },
    ]
)

lote_incremental = limpar_matriculas(lote_incremental_bruto, referencia_historica=matriculas_validas)
lote_incremental = lote_incremental[lote_incremental["data_matricula"].notna()].copy()

# Como o lote não trouxe pagamentos, criamos as medidas no grão da matrícula.
lote_incremental["quantidade_pagamentos"] = 0
lote_incremental["total_pago"] = 0.0
lote_incremental["ultimo_pagamento"] = pd.NaT
lote_incremental["canais_pagamento"] = "Nenhum"
lote_incremental["valor_liquido"] = (
    lote_incremental["mensalidade"] * (1 - lote_incremental["desconto_pct"] / 100)
).round(2)
lote_incremental["faixa_etaria"] = pd.cut(
    lote_incremental["idade"].astype(float),
    bins=[17, 25, 35, 50, 80],
    labels=["18–25", "26–35", "36–50", "51–80"],
)
lote_incremental["regiao"] = lote_incremental["cidade"].map(MAPA_REGIAO).fillna("Não informada")
lote_incremental["risco_financeiro"] = np.where(
    lote_incremental["status_matricula"].eq("Cancelada"), "Encerrado", "Alto"
)

carregar_dw(lote_incremental, tipo_carga="incremental")

# Reexecutamos o mesmo lote para demonstrar idempotência.
carregar_dw(lote_incremental, tipo_carga="incremental_reprocessada")

with sqlite3.connect(CAMINHO_DW) as conexao:
    total_fato = pd.read_sql_query("SELECT COUNT(*) AS quantidade FROM fato_matricula", conexao)
    auditoria = pd.read_sql_query("SELECT * FROM controle_carga ORDER BY carga_id", conexao)

display(total_fato)
display(auditoria)
print("O reprocessamento não duplicou as matrículas na tabela fato.")

## 10. Estrela e floco de neve

No esquema estrela atual, cidade e região permanecem dentro de `dim_aluno`. Em uma alternativa floco de neve, esses atributos poderiam formar `dim_localidade`, e `dim_aluno` armazenaria apenas `localidade_sk`.

| Aspecto | Estrela | Floco de neve |
|---|---|---|
| Dimensões | Mais desnormalizadas | Mais normalizadas |
| Junções | Menos junções | Mais junções |
| Consulta | Geralmente mais simples | Pode exigir mais conhecimento do modelo |
| Redundância | Maior | Menor |
| Manutenção | Simples para consumo | Útil para hierarquias compartilhadas |

Nenhum modelo é universalmente superior. A escolha depende do uso, desempenho, governança e manutenção.

## 11. Consumo analítico do Data Warehouse

In [ ]:
consulta_analitica = """
SELECT
    c.curso,
    f.status_matricula,
    COUNT(*) AS quantidade_matriculas,
    ROUND(SUM(f.valor_liquido), 2) AS valor_liquido_total,
    ROUND(SUM(f.total_pago), 2) AS total_pago,
    ROUND(AVG(f.desconto_pct), 2) AS desconto_medio_pct
FROM fato_matricula AS f
INNER JOIN dim_curso AS c ON c.curso_sk = f.curso_sk
GROUP BY c.curso, f.status_matricula
ORDER BY c.curso, f.status_matricula;
"""

with sqlite3.connect(CAMINHO_DW) as conexao:
    resultado = pd.read_sql_query(consulta_analitica, conexao)

display(resultado)

por_curso = resultado.groupby("curso", as_index=False)["valor_liquido_total"].sum().sort_values("valor_liquido_total")

fig, eixo = plt.subplots(figsize=(10, 5))
eixo.barh(por_curso["curso"], por_curso["valor_liquido_total"], color="#1F4E79")
eixo.set_title("Valor líquido das matrículas por curso")
eixo.set_xlabel("Valor líquido total (R$)")
eixo.set_ylabel("Curso")
eixo.grid(axis="x", color="#E5E7EB", linewidth=0.8)
eixo.spines[["top", "right"]].set_visible(False)

for posicao, valor in enumerate(por_curso["valor_liquido_total"]):
    eixo.text(valor + 100, posicao, f"R$ {valor:,.0f}".replace(",", "."), va="center")

plt.tight_layout()
plt.show()

## 12. Síntese técnica

| Conceito | Aplicação realizada |
|---|---|
| Limpeza e padronização | Nomes, cidades, cursos, status e e-mails |
| Nulos e duplicidades | Imputação sinalizada e versão mais recente |
| Conversão | Datas, números, moeda e categorias |
| Integração | Pagamentos agregados antes do `left join` |
| Regras de negócio | Idade, desconto, mensalidade e valor líquido |
| Enriquecimento | Faixa etária, região e risco financeiro |
| Carga completa | Reconstrução inicial do Data Warehouse |
| Carga incremental | `upsert`, marca d’água e idempotência |
| Modelagem dimensional | Dimensões, fato, grão e chaves substitutas |
| Auditoria | Tabela de controle de cargas |

# Atividade prática — Incremento do Data Warehouse

Uma nova unidade enviou um lote de matrículas. O arquivo contém registros novos, atualização de uma matrícula existente, nulos, desconto inválido, data impossível e inconsistências de formato.

## Entregas

1. Diagnosticar os problemas de qualidade;
2. Limpar e padronizar o lote reutilizando as funções da aula;
3. Separar registros válidos e registros de quarentena;
4. Aplicar as regras de negócio e os enriquecimentos;
5. Executar a carga incremental no Data Warehouse;
6. Reprocessar o mesmo lote e provar que não houve duplicidade;
7. Consultar a tabela de auditoria;
8. Criar uma consulta analítica e um gráfico;
9. Explicar em um parágrafo por que o grão da fato foi preservado.

**Tempo sugerido:** 60 minutos.  
**Organização:** duplas ou trios.

In [ ]:
# Arquivo entregue aos estudantes.
lote_atividade = pd.DataFrame(
    [
        {
            "matricula_id": 5027, "aluno_id": 1027, "nome": "  HELENA DANTAS ", "idade": 28,
            "email": "HELENA.DANTAS@EXEMPLO.COM ", "cidade": "RECIFE", "curso": "CD",
            "status_matricula": "ATIVO", "data_matricula": "26/09/2026",
            "mensalidade": "R$ 1.200,00", "desconto_pct": 10, "atualizado_em": "2026-09-26 09:00:00",
        },
        {
            "matricula_id": 5028, "aluno_id": 1028, "nome": "Igor Farias", "idade": None,
            "email": None, "cidade": "olinda", "curso": "IA",
            "status_matricula": "pendente", "data_matricula": "2026-09-26",
            "mensalidade": 1350, "desconto_pct": 75, "atualizado_em": "2026-09-26 09:30:00",
        },
        {
            "matricula_id": 5003, "aluno_id": 1003, "nome": "Carla Mendes", "idade": 36,
            "email": "carla.mendes@exemplo.com", "cidade": "Paulista", "curso": "Engenharia de Software",
            "status_matricula": "Ativa", "data_matricula": "2026-07-15",
            "mensalidade": 1050, "desconto_pct": 5, "atualizado_em": "2026-09-26 10:00:00",
        },
        {
            "matricula_id": 5029, "aluno_id": 1029, "nome": "Juliana Teixeira", "idade": 39,
            "email": "email_invalido", "cidade": "JABOATÃO", "curso": "ciencia de dados",
            "status_matricula": "Ativa", "data_matricula": "31/09/2026",
            "mensalidade": None, "desconto_pct": 0, "atualizado_em": "2026-09-26 10:30:00",
        },
        {
            "matricula_id": 5028, "aluno_id": 1028, "nome": "Igor Farias", "idade": 33,
            "email": "igor.farias@exemplo.com", "cidade": "Olinda", "curso": "Inteligência Artificial",
            "status_matricula": "Ativa", "data_matricula": "26/09/2026",
            "mensalidade": 1350, "desconto_pct": 15, "atualizado_em": "2026-09-26 11:00:00",
        },
    ]
)

caminho_atividade = PASTA_BRONZE / "lote_incremental_atividade.csv"
lote_atividade.to_csv(caminho_atividade, index=False)
display(lote_atividade)
print(f"Arquivo criado em: {caminho_atividade}")

## Espaço dos estudantes

In [ ]:
# TODO 1: leia caminho_atividade mantendo os tipos originais.
# TODO 2: use perfil_qualidade para diagnosticar o lote.
# TODO 3: aplique limpar_matriculas com referencia_historica=matriculas_validas.
# TODO 4: coloque datas inválidas e chaves ausentes em quarentena.
# TODO 5: crie quantidade_pagamentos, total_pago, valor_liquido, faixa_etaria, regiao e risco_financeiro.
# TODO 6: valide chave única, idade, desconto, mensalidade e valor líquido.
# TODO 7: execute carregar_dw(..., tipo_carga="atividade_incremental").
# TODO 8: execute novamente o mesmo lote e compare a contagem da fato antes e depois.
# TODO 9: consulte controle_carga e fato_matricula.
# TODO 10: produza uma consulta agregada e um gráfico.

print("Roteiro da atividade preparado. Complete os itens TODO.")

## Questões de reflexão

1. Por que a matrícula `5028` deve manter a versão das 11h?
2. Por que a matrícula `5029` deve ser encaminhada à quarentena?
3. Qual a diferença entre corrigir um formato e inventar um valor?
4. Por que os pagamentos precisam ser agregados antes da integração?
5. Como o `upsert` contribui para a idempotência?
6. Em que situação `dim_localidade` separada seria vantajosa?

## Avaliação — 10 pontos

| Critério | Pontos |
|---|---:|
| Diagnóstico e justificativa das decisões | 2,0 |
| Limpeza, conversões e regras de negócio | 2,0 |
| Quarentena e validações | 1,5 |
| Carga incremental e idempotência | 2,0 |
| Consulta, gráfico e explicação do grão | 2,5 |

## Gabarito do professor

> Remova esta seção antes de distribuir o notebook aos estudantes.

In [ ]:
# 1. Extração, diagnóstico e transformação.
atividade_bruta = pd.read_csv(caminho_atividade, dtype="object")
display(pd.DataFrame([perfil_qualidade("atividade", atividade_bruta, ["matricula_id"])]))

atividade_limpa = limpar_matriculas(atividade_bruta, referencia_historica=matriculas_validas)

mascara_rejeicao = (
    atividade_limpa["matricula_id"].isna()
    | atividade_limpa["aluno_id"].isna()
    | atividade_limpa["data_matricula"].isna()
)
atividade_quarentena = atividade_limpa[mascara_rejeicao].copy()
atividade_valida = atividade_limpa[~mascara_rejeicao].copy()

# 2. Medidas e enriquecimentos necessários à carga.
atividade_valida["quantidade_pagamentos"] = 0
atividade_valida["total_pago"] = 0.0
atividade_valida["ultimo_pagamento"] = pd.NaT
atividade_valida["canais_pagamento"] = "Nenhum"
atividade_valida["valor_liquido"] = (
    atividade_valida["mensalidade"] * (1 - atividade_valida["desconto_pct"] / 100)
).round(2)
atividade_valida["faixa_etaria"] = pd.cut(
    atividade_valida["idade"].astype(float),
    bins=[17, 25, 35, 50, 80],
    labels=["18–25", "26–35", "36–50", "51–80"],
)
atividade_valida["regiao"] = atividade_valida["cidade"].map(MAPA_REGIAO).fillna("Não informada")
atividade_valida["risco_financeiro"] = np.where(
    atividade_valida["status_matricula"].eq("Cancelada"), "Encerrado", "Alto"
)

# 3. Validações.
assert atividade_valida["matricula_id"].is_unique
assert atividade_valida["idade"].between(18, 80).all()
assert atividade_valida["desconto_pct"].between(0, 50).all()
assert atividade_valida["mensalidade"].ge(0).all()
assert atividade_valida["valor_liquido"].ge(0).all()

# 4. Demonstração da idempotência.
with sqlite3.connect(CAMINHO_DW) as conexao:
    quantidade_antes = conexao.execute("SELECT COUNT(*) FROM fato_matricula").fetchone()[0]

carregar_dw(
    atividade_valida,
    tipo_carga="atividade_incremental",
    registros_rejeitados=len(atividade_quarentena),
)
carregar_dw(
    atividade_valida,
    tipo_carga="atividade_reprocessada",
    registros_rejeitados=len(atividade_quarentena),
)

with sqlite3.connect(CAMINHO_DW) as conexao:
    quantidade_depois = conexao.execute("SELECT COUNT(*) FROM fato_matricula").fetchone()[0]
    auditoria_atividade = pd.read_sql_query(
        "SELECT * FROM controle_carga ORDER BY carga_id DESC LIMIT 4", conexao
    )

print(f"Linhas antes: {quantidade_antes}")
print(f"Linhas depois de carregar e reprocessar: {quantidade_depois}")
print(f"Novas matrículas válidas: {atividade_valida['matricula_id'].nunique()}")
display(atividade_quarentena)
display(auditoria_atividade)

In [ ]:
consulta_atividade = """
SELECT
    c.curso,
    COUNT(*) AS matriculas,
    ROUND(SUM(f.valor_liquido), 2) AS valor_liquido
FROM fato_matricula f
JOIN dim_curso c ON c.curso_sk = f.curso_sk
GROUP BY c.curso
ORDER BY valor_liquido DESC;
"""

with sqlite3.connect(CAMINHO_DW) as conexao:
    resultado_atividade = pd.read_sql_query(consulta_atividade, conexao)

display(resultado_atividade)

fig, eixo = plt.subplots(figsize=(9, 4))
eixo.barh(
    resultado_atividade["curso"],
    resultado_atividade["valor_liquido"],
    color="#D9A441",
)
eixo.set_title("Data Warehouse após a carga da atividade")
eixo.set_xlabel("Valor líquido (R$)")
eixo.set_ylabel("Curso")
eixo.grid(axis="x", color="#E5E7EB", linewidth=0.8)
eixo.spines[["top", "right"]].set_visible(False)
plt.tight_layout()
plt.show()

## Encerramento

O exercício conectou transformação, qualidade, regras de negócio, integração, carga e modelagem dimensional. A principal ideia é que um Data Warehouse confiável depende de decisões explícitas, validações executáveis, controle do grão, chaves consistentes e cargas auditáveis.

### Debate final

**Uma carga que terminou sem erro técnico pode ser considerada correta? Quais evidências adicionais seriam necessárias?**